<!--encabezado-grupo-->
> **Archivo:** `notebooks/consumidor_paso_a_paso.ipynb`  
> **Objetivo:** Notebook: consumidor_paso_a_paso  
> **Grupo del curso:** Fabio Durán - Gustavo Reyes - Ezequiel Muñoz  
> **Creación:** 20261110 1629 · **Última modificación:** 20261110 2013
<!--/encabezado-grupo-->

# Agente conversacional que consume al de viajes (A2A) · paso a paso

Tres pasos: **(1) configuración base**, **(2) agentes**, **(3) consumo**.

**Requisitos:** `uv run langgraph dev` corriendo en otra terminal (publica `agente_viajes` en `:2024`) y `GOOGLE_API_KEY` en `.env`.


## 1 · Configuración base

Cargar `.env`, obtener el `assistant_id` del agente de viajes, leer su Agent Card y mandarle un primer `message/send`.


In [ ]:
from dotenv import load_dotenv
load_dotenv()
from src.consumidor import a2a_cliente

aid = a2a_cliente.assistant_id()
print("assistant_id:", aid)

card = a2a_cliente.agent_card()
print("nombre:", card.get("name"))
print("descripción:", card.get("description"))
print("skills:", [s.get("name") for s in card.get("skills", [])])


In [ ]:
# Primer mensaje A2A, con un contextId nuevo para esta conversación
ctx = a2a_cliente.nuevo_context_id()
print(a2a_cliente.enviar_mensaje("Quiero un vuelo de Santiago a Lima el 2026-11-10", context_id=ctx))


## 2 · Agentes

El agente conversacional (`create_agent`, estado `messages`) con dos herramientas: `consultar_agente_viajes` (remota, A2A) y `precio_btc` (local dummy). El prompt está en `src/consumidor/prompts.py`.


In [ ]:
from src.consumidor.graph import graph, texto, HERRAMIENTAS
print("herramientas:", [h.name for h in HERRAMIENTAS])


## 3 · Consumo

Conversación de varios turnos en un mismo hilo (`thread_id`): mezcla una herramienta local (precio BTC) y la remota (viajes). El agente recuerda el contexto.


In [ ]:
hilo = {"configurable": {"thread_id": "nb-1"}}

def preguntar(t):
    r = graph.invoke({"messages": [{"role": "user", "content": t}]}, hilo)
    print(f"> {t}\n{texto(r['messages'][-1])}\n")

preguntar("Hola, ¿a cuánto está el bitcoin?")
preguntar("Busco un vuelo de Santiago a Lima el 2026-11-10")
preguntar("Y un hotel ahí por 2 noches")
preguntar("¿Me recordás qué te pregunté primero?")
